# Statistische Tests der Personalisierung
- erst friedman über alle modelle (schauen, ob es signifikante unterschiede gibt)
- dann danach paarweise zwischen den modellen

## Imports

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, wilcoxon

## Daten

In [2]:
subject_independent_folds = pd.read_csv("results/folds/rf_subject_independent_folds.csv")
normalized_folds = pd.read_csv("results/folds/rf_normalized_folds.csv")
rf_calibrated_k2_folds = pd.read_csv("results/folds/rf_calibrated_k2_folds.csv")
rf_calibrated_k4_folds = pd.read_csv("results/folds/rf_calibrated_k4_folds.csv")
rf_calibrated_k6_folds = pd.read_csv("results/folds/rf_calibrated_k6_folds.csv")
rf_calibrated_k8_folds = pd.read_csv("results/folds/rf_calibrated_k8_folds.csv")
rf_within_subject_folds = pd.read_csv("results/folds/rf_within_subject_folds.csv")  

## Daten Vorverabeitung
- within subject model, war nicht Loso sondern StratifiedKFold
- auf ein wert pro subject aggregieren

In [3]:
df_within_subject = pd.read_csv("results/folds/rf_within_subject_folds.csv")
rf_within_subject_per_subject = (
    df_within_subject.groupby("subject", as_index=False).agg(
        accuracy=("accuracy", "mean"),
        f1=("f1", "mean"),
        auc=("auc", "mean"),
        sensitivity=("sensitivity", "mean"),
        specificity=("specificity", "mean")
    )
)

## Daten Anschauen

In [4]:
# for models trained on Loso
df_subject_independent = pd.read_csv("results/folds/rf_subject_independent_folds.csv")
df_subject_independent.head()

,subject,accuracy,f1,auc,sensitivity,specificity
0,0,0.750,0.714286,0.843750,0.625,0.875
1,1,0.875,0.888889,0.984375,1.000,0.750
2,2,0.750,0.800000,0.890625,1.000,0.500
3,3,0.875,0.857143,0.906250,0.750,1.000
4,4,0.875,0.875000,0.953125,0.875,0.875


In [5]:
# for models trained within subjects aggregated per subject
df_within_subject = rf_within_subject_per_subject
df_within_subject.head()

,subject,accuracy,f1,auc,sensitivity,specificity
0,0,0.866667,0.760000,1.0,0.8,0.8
1,1,0.950000,0.933333,1.0,0.9,1.0
2,2,0.750000,0.626667,0.9,0.7,0.8
3,3,0.950000,0.933333,1.0,0.9,1.0
4,4,0.750000,0.766667,0.9,0.8,0.8


## Statistical Testing Fridman test
- hier testen, ob es zwischen allen ein signifikantes modell gibt

### Accuracy

In [6]:
subject_independant_accuracy = df_subject_independent["accuracy"].values
normalized_accuracy = normalized_folds["accuracy"].values
calibrated_k2_accuracy = rf_calibrated_k2_folds["accuracy"].values
calibrated_k4_accuracy = rf_calibrated_k4_folds["accuracy"].values
calibrated_k6_accuracy = rf_calibrated_k6_folds["accuracy"].values
calibrated_k8_accuracy = rf_calibrated_k8_folds["accuracy"].values
within_subject_accuracy = df_within_subject["accuracy"].values

# perform Friedman test
statistic, p_value = friedmanchisquare(
    subject_independant_accuracy,
    normalized_accuracy,
    calibrated_k2_accuracy,
    calibrated_k4_accuracy,
    calibrated_k6_accuracy,
    calibrated_k8_accuracy,
    within_subject_accuracy
)

print(f"Friedman test statistic: {statistic:.4f}, p-value: {p_value:.4f}")

Friedman test statistic: 26.0657, p-value: 0.0002


# Statistical test Wilcoxon signed rank test
- paarweise zwischen den modellen testen

### Accuracy

In [7]:
print("Subject independent vs. normalized:")
statistic, p_value = wilcoxon(subject_independant_accuracy, normalized_accuracy)
print(f"Wilcoxon test statistic: {statistic}, p-value: {p_value:.16f}")

Subject independent vs. normalized:
Wilcoxon test statistic: 87.0, p-value: 0.0114405232856846


In [8]:
for model in ["calibrated_k2", "calibrated_k4", "calibrated_k6", "calibrated_k8"]:
    print(f"Normalized vs. {model}:")
    model_accuracy = eval(f"{model}_accuracy")
    statistic, p_value = wilcoxon(normalized_accuracy, model_accuracy)
    print(f"Wilcoxon test statistic: {statistic}, p-value: {p_value:.16f}")
    print()

Normalized vs. calibrated_k2:
Wilcoxon test statistic: 173.0, p-value: 0.6987730528587763

Normalized vs. calibrated_k4:
Wilcoxon test statistic: 147.0, p-value: 0.4653415564508134

Normalized vs. calibrated_k6:
Wilcoxon test statistic: 131.0, p-value: 0.1619922148874360

Normalized vs. calibrated_k8:
Wilcoxon test statistic: 42.5, p-value: 0.0042334461586550

